In [1]:
import os
from pathlib import Path
import meeplemate
WORKSPACE_PATH = Path(meeplemate.__file__).parent.parent
os.chdir(WORKSPACE_PATH)

In [2]:
from IPython.display import display, Markdown
import importlib

from numpy import full
import dev_system
from meeplemate.config import GameService
importlib.reload(dev_system)
from langchain_openai import ChatOpenAI
from dev_system import areload, get_service
from meeplemate.component_system import factory

await areload(
    ["qa_service", "game_service", "chat_model", "chunk_search_service", "full_page_store"],
)
qa_service = get_service("qa_service")
game_service: GameService = get_service("game_service")
chat_model = get_service("chat_model")
chunk_search_service = get_service("chunk_search_service")
full_page_store = get_service("full_page_store")

Loading configuration from: config-dev.yaml
Loading configuration from: config-dev.yaml
Reloading system...
System reloaded


In [3]:
game_id = "warhammer_5th_edition"
manifest = await game_service.get_manifest(game_id)

## Queston Analysis

What is the user asking? What are the main rule interactions? What sort of question is it (e.g. rule lookup, rule interaction)?

In [5]:
from uuid import uuid4

from langchain_core.runnables import RunnableConfig
from meeplemate.qa_graph import (
    build_analyze_question_graph,
    QuestionAnalysisContext,
    create_question_analysis_state
)
from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.globals import set_debug

set_debug(False)

agent = build_analyze_question_graph(
    InMemorySaver(),
    chat_model,
)

context = QuestionAnalysisContext(
    manifest=manifest,
    chunk_search_service=chunk_search_service,
)
input = create_question_analysis_state(
    query="When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?"
)

config: RunnableConfig = {"configurable": {"thread_id": str(uuid4())}}

output = await agent.ainvoke(input, context=context, config=config)
# from pprint import pprint
# pprint(output)
# Display as Markdown
display(Markdown(output["question_analysis"]))

for interaction in output["direct_rule_interactions"]:
    print(interaction)

2026-02-02 19:10:15 [info     ] search_chunks called           search_terms=['Grail Knights', 'Green Dragon', 'break test', 'lose combat']
2026-02-02 19:10:22 [info     ] Retrieved results              query='Grail Knights' relevant_count=7 total_retrieved_count=7
2026-02-02 19:10:30 [info     ] Retrieved results              query='Green Dragon' relevant_count=2 total_retrieved_count=9
2026-02-02 19:10:47 [info     ] Retrieved results              query='break test' relevant_count=8 total_retrieved_count=8
2026-02-02 19:10:56 [info     ] Retrieved results              query='lose combat' relevant_count=5 total_retrieved_count=9
2026-02-02 19:10:56 [info     ] Analyzing question             document_count=22 query='When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?'


The user is asking whether Grail Knights, when they lose a combat against a Green Dragon, are required to take a Break test. This involves determining if the standard Break test mechanic applies in this specific scenario, particularly given that the Grail Knights have special rules (the Grail Virtue) that may exempt them from certain psychological effects like Panic tests. The query specifically focuses on whether losing combat triggers a Break test for Grail Knights, despite their special abilities.

Grail Virtue interacts with Break Test
Combat Result (Loss) interacts with Break Test


In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from meeplemate.qa_graph import (
    build_question_answer_graph,
    build_coordinating_agent_graph,
    build_analyze_question_graph
)

analyze_graph = build_analyze_question_graph(
    InMemorySaver(),
    chat_model,
)

qa_graph = build_question_answer_graph(
    checkpoint_saver=InMemorySaver(),
    chat_model=chat_model,
)

coord_graph = build_coordinating_agent_graph(
    checkpoint_saver=InMemorySaver(),
    analyze_question_agent=analyze_graph,
    game_agent=qa_graph,
)

In [ ]:
from meeplemate.qa_graph import GameAgentContext, CoordinationInputState
from langchain_core.runnables import RunnableConfig
from uuid import uuid4

input: CoordinationInputState = {
    "query": "When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?",
}

context: GameAgentContext = {
    "manifest": manifest,
    "chunk_search_service": chunk_search_service,
    "full_page_store": full_page_store
}

config: RunnableConfig = {"configurable": {"thread_id": str(uuid4())}}

result = await coord_graph.ainvoke(input, context=context, config=config)

display(Markdown(result["response"]))

for item in result["evidence"]:
    display(Markdown(item["content"]))

2026-02-04 05:33:45 [info     ] search_chunks called           search_terms=['Grail Knights', 'Green Dragon', 'combat', 'break test', 'losing combat']


/workspace/meeplemate/cassandra_util.py:39: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  return load(value) if value is not None else None


2026-02-04 05:33:54 [info     ] Retrieved results              query='Grail Knights' relevant_count=7 total_retrieved_count=7
2026-02-04 05:34:12 [info     ] Retrieved results              query='Green Dragon' relevant_count=2 total_retrieved_count=9
2026-02-04 05:34:22 [info     ] Retrieved results              query=combat relevant_count=5 total_retrieved_count=9
2026-02-04 05:34:39 [info     ] Retrieved results              query='break test' relevant_count=8 total_retrieved_count=8
2026-02-04 05:35:04 [info     ] Retrieved results              query='losing combat' relevant_count=5 total_retrieved_count=9
2026-02-04 05:35:04 [info     ] Analyzing question             document_count=25 query='When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?'
2026-02-04 05:35:07 [info     ] Question analysis result       analysis=AIMessage(content='The user is asking whether Grail Knights, when they lose a combat against a Green Dragon, 

**Yes, Grail Knights must take a Break test when they lose combat, even against a Green Dragon.**

The general rule for losing combat explicitly requires a Break test:

> The side that loses a combat must take a test to determine whether it stands and fights or turns tail and runs away This is called a Break test.
>
> (Warhammer Rulebook, p. 42)

This means that any unit, regardless of type, that loses a combat is subject to a Break test.

However, the Grail Knights have a special immunity:

> Grail Knights have the Grail Virtue; they have drunk from the sacred grail and are immune to psychology.
>
> (Bretonnia Army Book, p. 63)

This immunity applies specifically to psychological effects such as fear, panic, or terror, but does not extend to Break tests.

The rules clarify the distinction between these two types of tests:

> The two tests are quite separate. This is important because some bonuses apply specifically to Break tests and others apply specifically to psychology tests.
>
> (Warhammer Rulebook, p. 47)

This explicit separation confirms that immunity to psychology does not imply immunity to Break tests.

Therefore, since the Grail Knights’ immunity is limited to psychological effects and does not mention Break tests, they are still required to take a Break test when they lose combat. No exception overrides this rule, and no ability or card grants exemption from Break tests. The precedence hierarchy confirms that specific exceptions must explicitly name the mechanic to override it — which the Grail Virtue does not do.

Thus, Grail Knights must take a Break test when they lose combat, even when facing a Green Dragon.

TypeError: Markdown expects text, not {'rulebook_name': 'Warhammer Rulebook', 'page': 47, 'offset': -1, 'content': 'Players will immediately realise that a psychology test is taken in the same way as a Break test in hand- to- hand combat and uses the same characteristic, namely Leadership. However, a Break test is not a psychology test. The two tests are quite separate. This is important because some bonuses apply specifically to Break tests and others apply specifically to psychology tests.'}

In [11]:
from uuid import uuid4
from langchain_core.globals import set_debug

# set_debug(True)

input = {
    "manifest": manifest,
    "query": "How does Grail Virtue interact Break test?",
    "recursion_depth": 0,
    "evidence": [],
    "messages": [],
}

results = []
for _ in range(10):
    responses = await qa_service.abatch([input]*10, config={"configurable": {"thread_id": str(uuid4())}})

    for response in responses:
        result = response["response"].splitlines()[0]
        results.append(result)
        print(result)
        print()

# # Display response as markdown
# from IPython.display import Markdown, display
# display(Markdown(response["response"]))

2026-02-01 22:35:07 [info     ] search_chunks called           search_terms=['Grail Virtue', 'Break test']
2026-02-01 22:35:07 [info     ] search_chunks called           search_terms=['Grail Virtue', 'Break test']
2026-02-01 22:35:07 [info     ] search_chunks called           search_terms=['Grail Virtue', 'Break test']
2026-02-01 22:35:07 [info     ] search_chunks called           search_terms=['Grail Virtue', 'Break test']
2026-02-01 22:35:07 [info     ] search_chunks called           search_terms=['Grail Virtue', 'Break test']
2026-02-01 22:35:07 [info     ] search_chunks called           search_terms=['Grail Virtue', 'Break test']
2026-02-01 22:35:07 [info     ] search_chunks called           search_terms=['Grail Virtue', 'Break test']
2026-02-01 22:35:07 [info     ] search_chunks called           search_terms=['Grail Virtue', 'Break test']
2026-02-01 22:35:08 [info     ] search_chunks called           search_terms=['Grail Virtue', 'Break test']
2026-02-01 22:35:08 [info     ] searc

In [ ]:
# Right 95% of the time.
for result in sorted(results):
    print(result)

**Grail Knights are not exempt from Break tests, despite their Grail Virtue.**
**No, Grail Knights are not exempt from Break tests, even though they are unaffected by psychology rules.**
**No, Grail Knights are not exempt from Break tests, even though they have the Grail Virtue.**
**No, Grail Knights are not exempt from Break tests.**
**No, Grail Knights are not exempt from Break tests.**
**No, Grail Knights are not protected from Break tests by the Grail Virtue.**
**No, Grail Knights are not protected from Break tests by the Grail Virtue.**
**No, Grail Knights do not automatically avoid Break tests, even though they are unaffected by psychology.**
**No, Grail Knights do not bypass Break tests due to Grail Virtue.**
**No, Grail Knights do not bypass Break tests, even though they are granted immunity to psychology.**
**No, Grail Knights do not bypass Break tests, even though they possess the Grail Virtue.**
**No, Grail Knights do not gain immunity to Break tests from the Grail Virtue.**

In [13]:
for chunk in response["evidence"]:
    header = f"### From {chunk['rulebook_name']} page {chunk['page']}\n"
    display(Markdown(header + chunk["content"]))


### From Warhammer Rulebook page 47
Players will immediately realise that a psychology test is taken in the same way as a Break test in hand- to- hand combat and uses the same characteristic, namely Leadership. However, a Break test is not a psychology test. The two tests are quite separate. This is important because some bonuses apply specifically to Break tests and others apply specifically to psychology tests.